# Exercise Windowing & Labeling for IMU-based Exercise Classification

#### Capstone: comparing models that classify which arm exercise is being performed from dual-IMU (forearm + upper-arm) sensor data.

This is notebook 1 of the capstone, following the same structure as `capstone_example2_solution`: an EDA-style notebook that takes raw
recordings to a clean, labeled feature table, which later notebooks (`2.`, `3.`, ...) will use to train and compare classifiers.

## What this notebook does

The `MotionTracking` pipeline (`imu_preprocess.py` -> `imu_features.py` -> `imu_batch.py`) already turns raw accel/gyro streams into
per-*sample* motion features (pitch, elbow flexion, gyro/accel magnitude, jerk, ...). A classifier needs per-*window* features with a
label attached, and nothing in the repo produced that yet. This notebook:

1. Loads the session manifest (`manifest.csv`) that says which recording is which exercise.
2. Runs the existing feature-extraction pipeline over each recording.
3. Slices each recording into fixed-size overlapping time windows and aggregates each window into summary statistics (mean/std/min/max/rms per feature).
4. Saves the resulting table to `data/exercise_windows.csv` for the modeling notebooks.

The windowing/aggregation logic itself lives in `imu_dataset.py` (not copy-pasted into the notebook) so notebooks `2.` and `3.` can import it too.

## Standard package imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import imu_batch
import imu_dataset

plt.style.use("ggplot")
%matplotlib inline

REPO_ROOT = os.getcwd()

# Data Description

Each row of `manifest.csv` is one recording session:

- **session_id**: unique name for the recording
- **file**: path to the raw CSV, relative to the repo root
- **format**: which raw layout it's in -- `raw_dual` (native `mpu_dual_log.csv` / `read_mpu_dual.py` output) or `curl_csv`
  (the long-format `example_imu_data_bundle` layout, one row per sensor per timestamp)
- **label**: the exercise performed during that recording (blank = not yet labeled, excluded from the dataset)
- **subject**: who was wearing the sensors

Only one exercise is labeled so far (`bicep_curl`, from the synthetic example bundle). `mpu_dual_log.csv` is a real hardware capture with
no exercise recorded for it, so it's listed but skipped until it's labeled. **As more exercises get recorded, add a row to `manifest.csv`
and rerun this notebook -- nothing else needs to change.**

In [ ]:
manifest = imu_dataset.load_manifest("manifest.csv")
manifest

# Raw Session Inspection

Before windowing everything, look at one session end-to-end: the `bicep_curl` recording (`curl_csv` format, 50 Hz, ~20s, two sensors:
forearm and bicep/upper-arm).

In [ ]:
session = manifest[manifest.session_id == "curl_bicep_01"].iloc[0]
t_ms, a1, g1, a2, g2 = imu_dataset.load_curl_csv(session["file"])
print(f"{len(t_ms)} samples, {t_ms[-1]/1000:.1f}s, forearm=a1/g1, upper-arm=a2/g2")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
t_s = t_ms / 1000.0
axes[0].plot(t_s, np.linalg.norm(a1, axis=1), label="forearm |accel|")
axes[0].plot(t_s, np.linalg.norm(a2, axis=1), label="upper-arm |accel|")
axes[0].set_ylabel("m/s^2"); axes[0].legend(); axes[0].set_title("Raw accel magnitude")
axes[1].plot(t_s, np.linalg.norm(g1, axis=1), label="forearm |gyro|")
axes[1].plot(t_s, np.linalg.norm(g2, axis=1), label="upper-arm |gyro|")
axes[1].set_ylabel("deg/s"); axes[1].set_xlabel("time (s)"); axes[1].legend(); axes[1].set_title("Raw gyro magnitude")
plt.tight_layout()

# Per-Sample Feature Extraction

Run the existing pipeline (`imu_batch.process_batch`, the same bias/low-pass/complementary-filter math as the live `IMUPreprocessor` +
`FeatureEngine` classes) over the raw session. This is the step that turns raw accel/gyro into `elbow_flex`, motion magnitudes, and jerk.

In [ ]:
feat = imu_batch.process_batch(t_ms, a1, g1, a2, g2, gyro_units=imu_dataset.FORMAT_GYRO_UNITS["curl_csv"])
print(f"using C++ backend: {imu_batch.using_cpp_backend()}")
print(f"{len(feat['t_ms'])} feature samples")
pd.DataFrame(feat).head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
t_s = feat["t_ms"] / 1000.0
ax.plot(t_s, feat["pitch1"], label="pitch1 (forearm)")
ax.plot(t_s, feat["pitch2"], label="pitch2 (upper-arm)")
ax.plot(t_s, feat["elbow_flex"], label="elbow_flex", linewidth=2)
ax.set_xlabel("time (s)"); ax.set_ylabel("degrees"); ax.legend()
ax.set_title("Complementary-filter pitch & elbow flexion -- reps should be visible as periodic peaks")
plt.tight_layout()

The periodic rise and fall in `elbow_flex` is exactly the ~1-rep-per-4-seconds curl motion described in the example bundle's README --
this is the signal a window needs to capture whole, which drives the window size chosen below.

# Windowing

Each window needs to span at least one full rep so its aggregated stats (mean/std/min/max/rms of each feature) actually describe the
motion rather than a fragment of it. With curls at ~4s/rep, a **2s window with 1s hop (50% overlap)** captures roughly half a rep to a
full rep per window and gives multiple overlapping views of each rep -- a reasonable default that `imu_dataset.build_dataset` also uses.
Windows are placed by *time*, not sample count, so this still works if a sensor's rate drifts.

`imu_dataset.iter_windows` + `imu_dataset.aggregate_window` do the per-session work; `imu_dataset.build_dataset` drives it across every
labeled row in the manifest and stitches the sessions together.

Standardize data over usum for smoother distribution maybe, or 

In [2]:
WINDOW_SEC, HOP_SEC = 2.0, 1.0

windows_df = imu_dataset.build_dataset(
    "manifest.csv", repo_root=REPO_ROOT, window_sec=WINDOW_SEC, hop_sec=HOP_SEC,
)
print(windows_df.shape)
windows_df.head()

'curl_bicep_01' (bicep_curl): 18 windows
skip 'mpu_dual_log_01': no label in manifest
(18, 61)


,session_id,window_index,label,subject,n_samples,duration_s,pitch1_mean,pitch1_std,pitch1_min,pitch1_max,...,a1_mag_mean,a1_mag_std,a1_mag_min,a1_mag_max,a1_mag_rms,a2_mag_mean,a2_mag_std,a2_mag_min,a2_mag_max,a2_mag_rms
0,curl_bicep_01,0,bicep_curl,alex,100,1.98,25.274304,15.843496,0.097725,43.380132,...,9.792912,0.445455,9.142975,10.484659,9.803038,9.795264,0.134494,9.556197,10.059141,9.796187
1,curl_bicep_01,1,bicep_curl,alex,100,1.98,26.587797,17.124987,-12.978127,43.380132,...,10.233120,0.194093,9.764487,10.512930,10.234961,9.911849,0.073025,9.740300,10.059141,9.912118
2,curl_bicep_01,2,bicep_curl,alex,100,1.98,-6.780190,23.488111,-32.107360,37.273404,...,9.884321,0.432764,9.204985,10.512930,9.893790,9.797067,0.130527,9.599993,10.008350,9.797937
3,curl_bicep_01,3,bicep_curl,alex,100,1.98,-16.938720,14.816815,-32.107360,18.087154,...,9.432492,0.197966,9.124333,9.884467,9.434569,9.685176,0.051687,9.588289,9.795347,9.685313
4,curl_bicep_01,4,bicep_curl,alex,100,1.98,12.690308,22.052859,-27.799754,36.834379,...,9.767513,0.440144,9.124333,10.419489,9.777425,9.806381,0.134290,9.588289,10.052652,9.807301


# Label Distribution

With only one recorded exercise so far, this is necessarily a single bar -- the point is that this cell (and everything downstream) needs
no changes once more exercises are recorded and added to `manifest.csv`.

In [ ]:
counts = windows_df.label.value_counts()
counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", ax=ax)
ax.set_ylabel("# windows"); ax.set_title("Windows per exercise label")
plt.tight_layout()

# Sanity Check: Feature Summary

Quick look at the aggregated feature columns -- confirms no NaNs/infs leaked through and that the ranges look physically sensible
(e.g. `elbow_flex_mean` within a plausible degree range for a curl).

In [ ]:
print("any NaNs:", windows_df.isna().any().any())
windows_df.describe().T

# Save Windowed Dataset

Written to `data/exercise_windows.csv`, one row per window with `session_id` / `window_index` / `label` / `subject` plus the aggregated
feature columns -- the input table for the modeling notebooks.

In [ ]:
os.makedirs("data", exist_ok=True)
OUT_PATH = "data/exercise_windows.csv"
windows_df.to_csv(OUT_PATH, index=False)
print(f"wrote {len(windows_df)} windows x {len(windows_df.columns)} columns -> {OUT_PATH}")

# Next Steps

1. Record more exercises (`read_mpu_dual.py` -> `mpu_dual_log.csv`, or the equivalent for new sessions), add a row per session to
   `manifest.csv` with its `label`, and rerun this notebook -- `data/exercise_windows.csv` will pick up the new classes automatically.
2. Notebook `2.` : train and evaluate a classical model (e.g. random forest / gradient boosting) on the aggregated window features here.
3. Notebook `3.` : train a sequence model (e.g. an LSTM) directly on the per-sample feature windows (before aggregation) and compare
   against notebook `2.`'s results -- the actual model comparison this capstone is about.